# 实验 A：低维观测与高维状态重建

**项目目标：** 研究多个不完整观测能否重建隐藏状态，并区分：
1. 已知规则下的拼接；
2. 信息缺失时的估计；
3. 利用隐藏状态的结构约束进行重建；
4. “重建结果唯一”与“计算方法唯一”的区别。

**当前阶段：** 基础玩具实验与理论记录。此实验不是对大语言模型内部机制的直接验证，而是为后续研究“可辨识性、潜在变量与表征重建”建立基础。

## 0. 环境

依赖：
- `numpy`
- `matplotlib`

如需安装，在终端运行：

```bash
python -m pip install numpy matplotlib
```

Notebook 中的随机数生成使用 `numpy.random.default_rng`，固定种子便于复现实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 1. 生成隐藏的三维状态

每个隐藏状态为：

\[
z_i=(a_i,b_i,c_i)\in\mathbb{R}^3
\]

本实验先让三个坐标独立地服从区间 \([0,1)\) 上的均匀分布。

In [ ]:
n = 1000
Z = rng.random((n, 3))

print("Z shape:", Z.shape)
print("前 3 个隐藏状态：\n", Z[:3])

## 2. 构造三个二维视图

三个视图分别保留两个坐标：

\[
y^{(1)}=(a,b),\quad y^{(2)}=(a,c),\quad y^{(3)}=(b,c)
\]

每个视图单独看都缺少一个坐标，但多个视图合起来覆盖了全部坐标。这里我们**预先知道坐标对应关系**，因此这只是已知规则下的重建基线，还没有学习未知规则。

In [ ]:
view_ab = Z[:, [0, 1]]
view_ac = Z[:, [0, 2]]
view_bc = Z[:, [1, 2]]

print("view_ab shape:", view_ab.shape)
print("view_ac shape:", view_ac.shape)
print("view_bc shape:", view_bc.shape)

## 3. 根据已知规则重建

因为 `view_ab` 包含 a、b，而 `view_ac` 包含 c，所以只用前两个视图已经足够恢复每个三维点。第三个视图 `view_bc` 是冗余观测，可用于交叉一致性检查。

In [ ]:
Z_reconstructed = np.column_stack([
    view_ab[:, 0],  # a
    view_ab[:, 1],  # b
    view_ac[:, 1],  # c
])

errors = np.linalg.norm(Z - Z_reconstructed, axis=1)

print("样本数量:", len(Z))
print(f"最大重建误差: {errors.max():.12f}")
print(f"平均重建误差: {errors.mean():.12f}")
print(f"误差标准差: {errors.std():.12f}")

### 结果解释

在浮点精度范围内，重建误差为 0。这是因为观测完整保留了所有坐标，而且重建程序事先知道每个坐标位于哪个视图。

**当前结论：** 已知规则可以把互补视图拼回隐藏状态；这不等于我们已经从数据中学会了未知拼接规则。

## 4. 检查视图间的重复坐标

不同视图重复包含部分坐标。我们检查这些重复值是否一致。

In [ ]:
a_error = np.abs(view_ab[:, 0] - view_ac[:, 0])
b_error = np.abs(view_ab[:, 1] - view_bc[:, 0])
c_error = np.abs(view_ac[:, 1] - view_bc[:, 1])

print("a 的最大一致性误差:", a_error.max())
print("b 的最大一致性误差:", b_error.max())
print("c 的最大一致性误差:", c_error.max())

**解释：** 三个一致性误差均为 0。重复观测提供了交叉检查机会。但“一致”不必然代表“真实”：如果所有视图都一致地编码了错误状态，一致性检查仍可能通过。

## 5. 删除一个视图：为什么两个视图仍然足够？

只使用 `view_ab` 和 `view_ac`，仍然能够得到 a、b、c 三个坐标。因此重建仍然精确。`view_bc` 并非本设计中重建所必需。

In [ ]:
Z_two_views = np.column_stack([
    view_ab[:, 0],
    view_ab[:, 1],
    view_ac[:, 1],
])

errors_two_views = np.linalg.norm(Z - Z_two_views, axis=1)

print("只用 view_ab 和 view_ac：")
print("最大误差:", errors_two_views.max())
print("平均误差:", errors_two_views.mean())

## 6. 只保留 a、b：缺失的 c 暂时设为 0

因为原始数据中的 c 独立地从 \([0,1)\) 均匀抽取，单靠 a、b 无法知道每个样本的真实 c。把 c 设为 0 是一种估计策略，而不是恢复真实值。

In [ ]:
Z_incomplete = np.column_stack([
    view_ab[:, 0],
    view_ab[:, 1],
    np.zeros(len(view_ab)),
])

errors_incomplete = np.linalg.norm(Z - Z_incomplete, axis=1)

print("只用 view_ab，并把未知 c 设为 0：")
print("最大误差:", errors_incomplete.max())
print("平均误差:", errors_incomplete.mean())

### 为什么误差接近 0.5？

因为只有 c 这一维产生误差：

\[
e=\sqrt{(a-a)^2+(b-b)^2+(c-0)^2}=c
\]

而 \(c\sim U(0,1)\)，所以理论平均绝对/欧氏误差为：

\[
\mathbb{E}[e]=\mathbb{E}[c]=\frac12
\]

因此观察到平均误差约 0.497 是有限样本下接近理论值 0.5 的结果。最大误差接近 1，则是因为 1000 个均匀随机数中通常会有数值接近 1。

这不是说“恢复了 50% 的信息”，而是这个特定误差定义与数据分布共同产生了约 0.5 的平均误差。

## 7. 比较 c=0 与 c=0.5 两种估计

对均方误差（MSE）而言，如果没有其他信息且 \(c\sim U(0,1)\)，最优的常数预测是均值 0.5，而不是 0。

In [ ]:
c_true = Z[:, 2]
c_pred_zero = np.zeros(len(Z))
c_pred_mean = np.full(len(Z), 0.5)

for name, c_pred in [
    ("预测 c = 0", c_pred_zero),
    ("预测 c = 0.5", c_pred_mean),
]:
    abs_errors = np.abs(c_true - c_pred)
    mse = np.mean((c_true - c_pred) ** 2)
    rmse = np.sqrt(mse)
    print(f"\n{name}")
    print(f"平均绝对误差: {abs_errors.mean():.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")

### 理论解释：这不是正态分布专属性质

对任何具有有限二阶矩的随机变量 C 和常数 q：

\[
\mathbb{E}[(C-q)^2]
=\operatorname{Var}(C)+(\mathbb{E}[C]-q)^2
\]

因此，最小化期望平方误差的常数预测是：

\[
q^*=\mathbb{E}[C]
\]

对于均匀分布 \(U(0,1)\)，均值为 0.5。正态分布下，均值同样是平方误差损失的最优常数预测，但这里的数据并不服从正态分布。

均匀分布的理论值：
- 猜 0：\(\mathbb{E}[C^2]=1/3\)；
- 猜 0.5：\(\mathbb{E}[(C-0.5)^2]=1/12\)。

因此，预测 0.5 的期望 MSE 是预测 0 的四分之一。有限样本的实际值会略有波动。

## 8. 对照实验：独立 c 与结构化 c

我们对比两种数据生成规则：

1. **独立情形：** c 与 a、b 独立随机生成。只观察 a、b 时，无法精确恢复每个样本的 c。
2. **结构化情形：** c=a+b。只要观察 a、b，就可以按已知规则精确恢复 c。

这个对照说明：可恢复性不仅取决于观测数量，也取决于隐藏状态是否存在可利用的结构。

In [ ]:
rng_compare = np.random.default_rng(123)
n_compare = 10000

a = rng_compare.random(n_compare)
b = rng_compare.random(n_compare)

c_independent = rng_compare.random(n_compare)
c_structured = a + b

# 独立 c：没有 a、b 之外的预测信息，使用总体均值 0.5 作为预测
c_pred_independent = np.full(n_compare, 0.5)
mse_independent = np.mean((c_independent - c_pred_independent) ** 2)

# 结构化 c：根据已知关系 c = a + b 直接重建
c_reconstructed_structured = a + b
mse_structured = np.mean((c_structured - c_reconstructed_structured) ** 2)

print("独立 c 的预测 MSE:", mse_independent)
print("结构化 c 的重建 MSE:", mse_structured)
print("独立情形理论 MSE:", 1 / 12)

## 9. 数学结论：线性观测与可辨识性

对于线性观测：

\[
y=Az
\]

若两个状态 \(z_1,z_2\) 产生同一观测，则：

\[
Az_1=Az_2\implies A(z_1-z_2)=0
\]

因此，唯一性取决于矩阵 A 的零空间。若要对任意 \(n\) 维状态保证唯一恢复，线性映射 A 必须满足：

\[
\operatorname{rank}(A)=n
\]

本实验只观测 a、b 时，观测矩阵可以写成：

\[
A=\begin{bmatrix}1&0&0\\0&1&0\end{bmatrix}
\]

其秩为 2，小于隐藏状态维度 3，因此 c 方向不可辨识，存在无穷多个与观测一致的状态。

如果状态受到额外约束（例如 c=a+b），则即使没有直接观测 c，也可能唯一恢复。更一般地，需要判断观测映射在允许的隐藏状态集合上是否为单射。

### 零误差不意味着计算方法唯一

即使隐藏状态唯一，仍可以有许多不同算法得到相同答案。必须区分：
- 解是否唯一；
- 算法是否唯一；
- 训练数据上的误差是否为 0；
- 新样本上的误差是否为 0；
- 对整个允许状态空间是否都能保证零误差。

## 10. 当前实验结论与局限

### 已验证
- 三个二维视图共同包含完整坐标时，可以按已知规则精确重建。
- 本实验中两个视图已经覆盖全部坐标，第三个视图主要提供冗余和一致性检查。
- 只观察 a、b 并将 c 设为 0，平均误差约为 0.5，因为误差恰好等于均匀随机变量 c。
- 对平方误差，常数预测的最优值是变量的均值；本实验的均匀分布均值为 0.5，而非因为数据服从正态分布。
- 当 c 独立于 a、b 时，仅靠 a、b 无法精确预测每个样本的 c；当存在已知结构 c=a+b 时，可以精确恢复。
- 线性观测下，任意状态唯一恢复的条件是观测矩阵满列秩；若隐藏状态有额外结构，可在更受限的状态集合上获得唯一性。

### 尚未验证
- 如何从数据中学习未知的观测/拼接规则。
- 在未知线性变换或非线性变换下，何时能辨识隐藏结构。
- 如何把这些结论谨慎地迁移到开源语言模型的内部表征研究。

### 下一步建议
实验 B：对各视图施加未知线性变换，利用成对样本和训练/测试划分研究能否恢复对应关系，以及何时出现多个同样符合训练数据的候选规则。